<a href="https://colab.research.google.com/github/ChanZH0525/Salary-Prediction-Prototype-Model/blob/main/new_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import set_seed

SEED =42

def set_all_seeds(seed: int):
    """Seed Python, NumPy, torch (CPU + CUDA) and transformers.
    Call at the start of every run, before model init and DataLoader creation."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

In [ ]:
from google.colab import drive
import pandas as pd

# Mount Google Drive to access your files
drive.mount('/content/drive')

# Define the path to your CSV file within your Google Drive folder
# Replace 'My Drive/Your_Folder/df_labeled_cleaned_with_exp.csv' with your actual path
file_path = '/content/drive/My Drive/UM MAI/df_labeled_cleaned_with_exp.csv'

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display basic information and the first few rows of the DataFrame
df.info()
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28960 entries, 0 to 28959
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   job_id              28960 non-null  int64  
 1   company             28960 non-null  object 
 2   type                28960 non-null  object 
 3   salary              28960 non-null  object 
 4   listingDate         28960 non-null  object 
 5   descriptions_clean  28960 non-null  object 
 6   desc_word_len       28960 non-null  int64  
 7   job_title_clean     28960 non-null  object 
 8   role_clean          28960 non-null  object 
 9   state               28960 non-null  object 
 10  salary_range        28960 non-null  object 
 11  salary_mid          28960 non-null  float64
 12  salary_width        28960 non-null  float64
 13  salary_min          28

,job_id,company,type,salary,listingDate,descriptions_clean,desc_word_len,job_title_clean,role_clean,state,salary_range,salary_mid,salary_width,salary_min,salary_max,category_celan,subcategory_celan,exp_min,exp_max,experience_raw
0,84094132,VERONA BAKE (M) SDN BHD,Full time,"RM 3,500 – RM 4,500 per month",8/5/2025 10:17,We are a beloved bakery brand known for our fr...,296,Marketing Executive,marketing executive,Selangor,"3,500 - 4,500",4000.0,1000.0,3500.0,4500.0,marketing & communications,marketing communications,1.0,2.0,1 to 2
1,84090140,SRG Asia Pacific Sdn. Bhd.,Full time,"RM 4,000 – RM 4,500 per month",8/5/2025 10:10,POSITION SUMMARY We are seeking a detail-orien...,715,Customer Service Administrator | Import Export,customer service administrator,Selangor,"4,000 - 4,500",4250.0,500.0,4000.0,4500.0,"manufacturing, transport & logistics",import/export & customs,NaN,NaN,Unspecified
2,84092513,Kanry Search,Full time,"RM 4,000 – RM 6,000 per month",8/5/2025 9:25,JOB RESPONSIBILITIES Collaborate with the Plan...,291,Buyer,buyer,Johor,"4,000 - 6,000",5000.0,2000.0,4000.0,6000.0,"manufacturing, transport & logistics","purchasing, procurement & inventory",NaN,NaN,Unspecified
3,84091988,TT Freight Group Sdn Bhd,Full time,"RM 3,000 – RM 4,500 per month",8/5/2025 9:06,Responsibility Building and maintaining relati...,219,Sales Executive,sales executive,Selangor,"3,000 - 4,500",3750.0,1500.0,3000.0,4500.0,sales,sales representatives/consultants,0.0,0.0,0
4,84091161,VILLAGE GROCER HOLDINGS SDN BHD,Full time,"RM 1,700 – RM 2,400 per month",8/5/2025 8:45,Job Description Assistant Baker Strictly follo...,196,Assistant Baker (Central Bakery Kitchen),bakery assistant,Selangor,"1,700 - 2,400",2050.0,700.0,1700.0,2400.0,trades & services,bakers & pastry chefs,1.0,3.0,1 to 3


In [ ]:

df = pd.read_csv('/content/df_labeled_cleaned_with_exp.csv')
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28960 entries, 0 to 28959
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   job_id              28960 non-null  int64  
 1   company             28960 non-null  object 
 2   type                28960 non-null  object 
 3   salary              28960 non-null  object 
 4   listingDate         28960 non-null  object 
 5   descriptions_clean  28960 non-null  object 
 6   desc_word_len       28960 non-null  int64  
 7   job_title_clean     28960 non-null  object 
 8   role_clean          28960 non-null  object 
 9   state               28960 non-null  object 
 10  salary_range        28960 non-null  object 
 11  salary_mid          28960 non-null  float64
 12  salary_width        28960 non-null  float64
 13  salary_min          28960 non-null  float64
 14  salary_max          28960 non-null  float64
 15  category_celan      28960 non-null  object 
 16  subc

,job_id,company,type,salary,listingDate,descriptions_clean,desc_word_len,job_title_clean,role_clean,state,salary_range,salary_mid,salary_width,salary_min,salary_max,category_celan,subcategory_celan,exp_min,exp_max,experience_raw
0,84094132,VERONA BAKE (M) SDN BHD,Full time,"RM 3,500 – RM 4,500 per month",8/5/2025 10:17,We are a beloved bakery brand known for our fr...,296,Marketing Executive,marketing executive,Selangor,"3,500 - 4,500",4000.0,1000.0,3500.0,4500.0,marketing & communications,marketing communications,1.0,2.0,1 to 2
1,84090140,SRG Asia Pacific Sdn. Bhd.,Full time,"RM 4,000 – RM 4,500 per month",8/5/2025 10:10,POSITION SUMMARY We are seeking a detail-orien...,715,Customer Service Administrator | Import Export,customer service administrator,Selangor,"4,000 - 4,500",4250.0,500.0,4000.0,4500.0,"manufacturing, transport & logistics",import/export & customs,NaN,NaN,Unspecified
2,84092513,Kanry Search,Full time,"RM 4,000 – RM 6,000 per month",8/5/2025 9:25,JOB RESPONSIBILITIES Collaborate with the Plan...,291,Buyer,buyer,Johor,"4,000 - 6,000",5000.0,2000.0,4000.0,6000.0,"manufacturing, transport & logistics","purchasing, procurement & inventory",NaN,NaN,Unspecified
3,84091988,TT Freight Group Sdn Bhd,Full time,"RM 3,000 – RM 4,500 per month",8/5/2025 9:06,Responsibility Building and maintaining relati...,219,Sales Executive,sales executive,Selangor,"3,000 - 4,500",3750.0,1500.0,3000.0,4500.0,sales,sales representatives/consultants,0.0,0.0,0
4,84091161,VILLAGE GROCER HOLDINGS SDN BHD,Full time,"RM 1,700 – RM 2,400 per month",8/5/2025 8:45,Job Description Assistant Baker Strictly follo...,196,Assistant Baker (Central Bakery Kitchen),bakery assistant,Selangor,"1,700 - 2,400",2050.0,700.0,1700.0,2400.0,trades & services,bakers & pastry chefs,1.0,3.0,1 to 3


In [ ]:
# Manual correction: raw "RM 7000 - RM 11 000" was misparsed due to a space
# used as a thousands separator in "11 000" (instead of comma/no separator),
# causing the parser to truncate it to 11 and misassign min/max as 0/7000.
# Confirmed via df['salary'].str.contains(r'\d \d{3}\b') that this is the
# only row in the dataset affected by this specific space-separator pattern.

mask = df["job_id"] ==  74018456 # fill in the actual job_id for this row

df.loc[mask, "salary_min"] = 7000.0
df.loc[mask, "salary_max"] = 11000.0
df.loc[mask, "salary_mid"] = (7000.0 + 11000.0) / 2
df.loc[mask, "salary_width"] = 11000.0 - 7000.0
df.loc[mask, "salary_range"] = "7,000 - 11,000"

print(df.loc[mask, ["salary", "salary_min", "salary_max", "salary_mid", "salary_width", "salary_range"]])

                    salary  salary_min  salary_max  salary_mid  salary_width  \
28137  RM 7000 - RM 11 000      7000.0     11000.0      9000.0        4000.0   

         salary_range  
28137  7,000 - 11,000  


In [ ]:
# Stage 1, step 9 (revised): upper salary_mid bound lowered from RM 150,000
# to RM 20,000. Bracket-level row counts (fit on train, RM 2,000 bins)
# showed brackets above RM 20,000 routinely contain 0-9 rows per split,
# too sparse to support reliable model training or evaluation. This trims
# ~147 rows (~0.5% of the dataset) previously retained under the original
# [RM 500, RM 150,000] bound.
SALARY_MID_UPPER_BOUND = 20000
MID_COL = "salary_mid"

before_n = len(df)
df = df[df[MID_COL] <= SALARY_MID_UPPER_BOUND].reset_index(drop=True)
print(f"Dropped {before_n - len(df)} rows with salary_mid > RM {SALARY_MID_UPPER_BOUND:,} "
      f"({before_n} -> {len(df)})")

Dropped 138 rows with salary_mid > RM 20,000 (28960 -> 28822)


In [ ]:
# ------------------------------------------------------------
# 1. BUILD STRUCTURED TEXT
#    TITLE | CATEGORY | SUBCATEGORY | LOCATION | ROLE | TYPE | EXPERIENCE | DESCRIPTION
#    DESCRIPTION is last so head truncation only cuts the description tail.
# ------------------------------------------------------------

def build_structured_text(row):
    parts = [
        f"TITLE: {row['job_title_clean']}",
        f"CATEGORY: {row['category_celan']}",
        f"SUBCATEGORY: {row['subcategory_celan']}",
        f"LOCATION: {row['state']}",
        f"ROLE: {row['role_clean']}",
        f"TYPE: {row['type']}",
        f"EXPERIENCE: {str(row['experience_raw']).lower()}",
    ]
    structured = " | ".join(parts)
    return f"{structured} | DESCRIPTION: {row['descriptions_clean']}"

df['structured_text'] = df.apply(build_structured_text, axis=1)

#FIELD_TAGS = ["TITLE", "CATEGORY", "SUBCATEGORY", "LOCATION", "ROLE", "TYPE", "DESCRIPTION"]
FIELD_TAGS = ["TITLE", "CATEGORY", "SUBCATEGORY", "LOCATION", "ROLE", "TYPE", "EXPERIENCE", "DESCRIPTION"]
# Sanity checks: no missing tagged fields leaked in as literal "nan"
tagged_part = df['structured_text'].str.split(" \\| DESCRIPTION:", n=1, regex=True).str[0]
assert not tagged_part.str.contains(r"\bnan\b", case=False, regex=True).any(), "NaN leaked into tagged fields"
print(df['structured_text'].iloc[0][:400])

TITLE: Marketing Executive | CATEGORY: marketing & communications | SUBCATEGORY: marketing communications | LOCATION: Selangor | ROLE: marketing executive | TYPE: Full time | DESCRIPTION: We are a beloved bakery brand known for our freshly baked goods and warm customer service. As we grow, we re looking for a creative and driven Marketing Executive to help boost our brand presence, promote our pro


In [ ]:
# ------------------------------------------------------------
# 2. TARGET TRANSFORM — predicting (median, width).
#    salary_mid/salary_width already provided in the cleaned dataset.
#    log1p-transform both targets for training.
# ------------------------------------------------------------

assert (df['salary_width'] >= 0).all(), "Negative widths found — investigate before proceeding."

df['log_mid'] = np.log1p(df['salary_mid'])
df['log_width'] = np.log1p(df['salary_width'])

In [ ]:
DATE_COLUMN = "listingDate"

# Format is D/M/YYYY H:MM (e.g. "8/5/2025 10:17"), day-first. Must be explicit,
# since day <= 12 rows are ambiguous and pandas silently mis-infers month-first.
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], format="%d/%m/%Y %H:%M", errors="raise")

# Deterministic order: timestamp, then job_id as tie-breaker (stable sort)
df = df.sort_values([DATE_COLUMN, "job_id"], kind="mergesort").reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train dates: {train_df[DATE_COLUMN].min().date()} to {train_df[DATE_COLUMN].max().date()}")
print(f"Val dates:   {val_df[DATE_COLUMN].min().date()} to {val_df[DATE_COLUMN].max().date()}")
print(f"Test dates:  {test_df[DATE_COLUMN].min().date()} to {test_df[DATE_COLUMN].max().date()}")

assert len(train_df) + len(val_df) + len(test_df) == n
assert not (set(train_df["job_id"]) & set(val_df["job_id"]))
assert not (set(val_df["job_id"]) & set(test_df["job_id"]))
assert not (set(train_df["job_id"]) & set(test_df["job_id"]))

Train: 20175 | Val: 4323 | Test: 4324
Train dates: 2024-01-27 to 2024-05-31
Val dates:   2024-05-31 to 2025-03-11
Test dates:  2025-03-11 to 2025-05-08


In [ ]:
# ------------------------------------------------------------
# 4. TOKENIZATION (BERT-base-cased, 512-token limit, dynamic padding)
# ------------------------------------------------------------

from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "bert-base-cased"
#MODEL_NAME = "jjzha/jobbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(texts):
    return tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding=False,  # padding deferred to the data collator, per-batch
    )

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class SalaryRangeDataset(torch.utils.data.Dataset):
    """labels is a 2-vector: [log_mid, log_width] per row."""
    def __init__(self, texts, log_mid, log_width):
        self.encodings = tokenize_function(list(texts))
        self.labels = list(zip(log_mid, log_width))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = SalaryRangeDataset(train_df['structured_text'], train_df['log_mid'], train_df['log_width'])
val_dataset = SalaryRangeDataset(val_df['structured_text'], val_df['log_mid'], val_df['log_width'])
test_dataset = SalaryRangeDataset(test_df['structured_text'], test_df['log_mid'], test_df['log_width'])

In [ ]:
# ------------------------------------------------------------
# 5. MODEL (regression head, 2 outputs: log_mid, log_width)
# ------------------------------------------------------------

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="regression",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import EarlyStoppingCallback


def _r2_safe(true, pred):
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - np.mean(true)) ** 2)
    return 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan


def _reconstruct(predictions, split_df):
    """Log-space predictions [n,2] -> RM-scale arrays.
    True mid/width/min/max come from the dataframe (same as the XGBoost and
    fusion runs), NOT from expm1(labels), so Hit uses the actual quoted bounds."""
    pred_log_mid, pred_log_width = predictions[:, 0], predictions[:, 1]

    pred_mid = np.expm1(pred_log_mid)
    pred_width = np.maximum(np.expm1(pred_log_width), 0)
    pred_min = np.maximum(pred_mid - pred_width / 2, 0)
    pred_max = pred_min + pred_width

    return dict(
        pred_mid=pred_mid, pred_width=pred_width, pred_min=pred_min, pred_max=pred_max,
        pred_log_mid=pred_log_mid, pred_log_width=pred_log_width,
        true_mid=split_df["salary_mid"].values.astype(float),
        true_width=split_df["salary_width"].values.astype(float),
        true_min=split_df["salary_min"].values.astype(float),
        true_max=split_df["salary_max"].values.astype(float),
        true_log_mid=split_df["log_mid"].values.astype(float),
        true_log_width=split_df["log_width"].values.astype(float),
    )


def _composite_parts(r):
    """Hit, width ratio and per-row composite, matching the fusion notebook:
    composite_i = hit_i * min(WR_i, 1/WR_i); zero-width true rows have WR = NaN
    and are dropped from the composite mean (np.nanmean)."""
    hit = (r["pred_mid"] >= r["true_min"]) & (r["pred_mid"] <= r["true_max"])

    nz = r["true_width"] > 0
    wr = np.full(len(hit), np.nan)
    wr[nz] = r["pred_width"][nz] / r["true_width"][nz]
    with np.errstate(divide="ignore", invalid="ignore"):
        calib = np.minimum(wr, 1.0 / wr)
    composite_row = hit.astype(float) * calib
    return hit, wr, composite_row


def make_compute_metrics(split_df):
    """Returns a compute_metrics function bound to one split (val_df or test_df).
    Relies on the eval dataset being built from split_df in the same row order
    and the eval dataloader not shuffling (Trainer default)."""

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        assert len(predictions) == len(split_df), "Prediction count != split size"
        r = _reconstruct(predictions, split_df)
        hit, wr, composite_row = _composite_parts(r)

        # Layer 1: component accuracy (RM scale + log scale)
        mid_mae = mean_absolute_error(r["true_mid"], r["pred_mid"])
        mid_rmse = np.sqrt(mean_squared_error(r["true_mid"], r["pred_mid"]))
        mid_r2 = _r2_safe(r["true_mid"], r["pred_mid"])
        mid_log_r2 = _r2_safe(r["true_log_mid"], r["pred_log_mid"])

        width_mae = mean_absolute_error(r["true_width"], r["pred_width"])
        width_rmse = np.sqrt(mean_squared_error(r["true_width"], r["pred_width"]))
        width_r2 = _r2_safe(r["true_width"], r["pred_width"])
        width_log_r2 = _r2_safe(r["true_log_width"], r["pred_log_width"])

        # Layer 2: Hit Rate + DtB-MAE (misses only)
        hit_rate = float(np.mean(hit))
        miss_mask = ~hit
        if miss_mask.any():
            dist_below = np.maximum(r["true_min"] - r["pred_mid"], 0)
            dist_above = np.maximum(r["pred_mid"] - r["true_max"], 0)
            dtb_mae = float(np.mean(np.maximum(dist_below, dist_above)[miss_mask]))
        else:
            dtb_mae = 0.0

        # Layer 3: width ratio (median, zero-width true rows excluded) + composite
        width_ratio_median = float(np.nanmedian(wr)) if np.isfinite(wr).any() else np.nan
        composite = float(np.nanmean(composite_row))

        return {
            "mid_mae": mid_mae, "mid_rmse": mid_rmse, "mid_r2": mid_r2, "mid_log_r2": mid_log_r2,
            "width_mae": width_mae, "width_rmse": width_rmse, "width_r2": width_r2, "width_log_r2": width_log_r2,
            "hit_rate": hit_rate, "dtb_mae": dtb_mae,
            "width_ratio_median": width_ratio_median,
            "composite": composite,
        }

    return compute_metrics


def stratified_metrics(predictions, split_df, bracket_bins=None, bracket_labels=None,
                       width_bucket_edges=(1000, 1500)):
    """
    Post-hoc Layer 1/2/3 breakdown by fixed RM2,000 salary bracket, and Hit Rate
    by true-width bucket. Run on final test-set predictions, not during training.
    Width buckets use FIXED edges (RM1,000 / RM1,500) to match the XGBoost and
    fusion tables: narrow <= 1,000 < medium <= 1,500 < wide, plus zero_width.
    """
    if bracket_bins is None:
        bracket_bins = list(range(0, 20001, 2000)) + [np.inf]
    if bracket_labels is None:
        bracket_labels = [f"{bracket_bins[i]//1000}k-{bracket_bins[i+1]//1000}k"
                          for i in range(len(bracket_bins) - 2)] + ["20k+"]

    r = _reconstruct(predictions, split_df)
    hit, wr, composite_row = _composite_parts(r)
    miss_mask = ~hit
    dist_below = np.maximum(r["true_min"] - r["pred_mid"], 0)
    dist_above = np.maximum(r["pred_mid"] - r["true_max"], 0)
    dist_to_bound = np.maximum(dist_below, dist_above)

    df_eval = pd.DataFrame({
        "true_mid": r["true_mid"], "pred_mid": r["pred_mid"],
        "true_width": r["true_width"], "pred_width": r["pred_width"],
        "abs_err_mid": np.abs(r["pred_mid"] - r["true_mid"]),
        "abs_err_width": np.abs(r["pred_width"] - r["true_width"]),
        "hit": hit, "miss": miss_mask, "dist_to_bound": dist_to_bound,
        "width_ratio": wr, "composite_row": composite_row,
    })
    df_eval["salary_bracket"] = pd.cut(df_eval["true_mid"], bins=bracket_bins, labels=bracket_labels)

    def _agg(g):
        miss_g = g[g["miss"]]
        return pd.Series({
            "mid_mae": g["abs_err_mid"].mean(),
            "width_mae": g["abs_err_width"].mean(),
            "hit_rate": g["hit"].mean(),
            "dtb_mae": miss_g["dist_to_bound"].mean() if len(miss_g) else 0.0,
            "width_ratio_median": g["width_ratio"].median() if g["width_ratio"].notna().any() else np.nan,
            "composite": g["composite_row"].mean() if g["composite_row"].notna().any() else np.nan,
            "n": len(g),
        })

    grouped = df_eval.groupby("salary_bracket", observed=True).apply(_agg)
    overall = pd.DataFrame([_agg(df_eval)], index=["Overall"])
    bracket_table = pd.concat([grouped, overall])

    # True-width buckets (fixed edges)
    e1, e2 = width_bucket_edges
    zero_mask = df_eval["true_width"] == 0
    df_eval["width_bucket"] = "zero_width"
    df_eval.loc[~zero_mask & (df_eval["true_width"] <= e1), "width_bucket"] = "narrow"
    df_eval.loc[~zero_mask & (df_eval["true_width"] > e1) & (df_eval["true_width"] <= e2), "width_bucket"] = "medium"
    df_eval.loc[~zero_mask & (df_eval["true_width"] > e2), "width_bucket"] = "wide"

    def _agg_width_bucket(g):
        miss_g = g[g["miss"]]
        return pd.Series({
            "hit_rate": g["hit"].mean(),
            "dtb_mae": miss_g["dist_to_bound"].mean() if len(miss_g) else 0.0,
            "n": len(g),
        })

    width_bucket_table = (df_eval.groupby("width_bucket", observed=True)
                          .apply(_agg_width_bucket)
                          .reindex(["narrow", "medium", "wide", "zero_width"]))

    return bracket_table, width_bucket_table


# Usage (once train/val/test frames exist):
# val_compute_metrics  = make_compute_metrics(val_df)    # for Trainer(compute_metrics=...)
# test_compute_metrics = make_compute_metrics(test_df)   # for final test evaluation

In [ ]:
# ------------------------------------------------------------
# 7. PRINT REPORT — formats stratified_metrics() output to match the
#    reference XGBoost report. Width bucket edges are fit on train and
#    passed in explicitly for val/test to avoid leakage.
# ------------------------------------------------------------

def fit_width_bucket_edges(train_df_):
    """Fit tercile edges on TRAIN true salary_width only. Call once, reuse for val/test."""
    w = train_df_["salary_width"].values.astype(float)
    return np.quantile(w[w > 0], [1/3, 2/3])


def print_report(predictions, split_df, model_name, split_name, width_bucket_edges,
                 bracket_bins=None, bracket_labels=None):
    bracket_table, width_bucket_table = stratified_metrics(
        predictions, split_df,
        bracket_bins=bracket_bins, bracket_labels=bracket_labels,
        width_bucket_edges=width_bucket_edges,
    )
    r = _reconstruct(predictions, split_df)
    hit, wr, composite_row = _composite_parts(r)
    overall = bracket_table.loc["Overall"]

    mid_mae = mean_absolute_error(r["true_mid"], r["pred_mid"])
    mid_rmse = np.sqrt(mean_squared_error(r["true_mid"], r["pred_mid"]))
    mid_r2 = _r2_safe(r["true_mid"], r["pred_mid"])
    mid_log_r2 = _r2_safe(r["true_log_mid"], r["pred_log_mid"])
    width_mae = mean_absolute_error(r["true_width"], r["pred_width"])
    width_rmse = np.sqrt(mean_squared_error(r["true_width"], r["pred_width"]))
    width_r2 = _r2_safe(r["true_width"], r["pred_width"])
    width_log_r2 = _r2_safe(r["true_log_width"], r["pred_log_width"])

    ratios = wr[np.isfinite(wr)]
    ratio_median = np.median(ratios)
    ratio_q1, ratio_q3 = np.percentile(ratios, [25, 75])
    composite = np.nanmean(composite_row)

    zero_mask = r["true_width"] == 0
    n_zero = int(zero_mask.sum())
    zero_pred_width_median = np.median(r["pred_width"][zero_mask]) if n_zero > 0 else float("nan")

    print("=" * 60)
    print(f"{model_name} — {split_name} (3-Layer Framework, RM2k brackets)")
    print("=" * 60)

    print("\n--- Overall ---")
    print(f"  Mid:   MAE={mid_mae:,.0f}  RMSE={mid_rmse:,.0f}  R2={mid_r2:.3f}  R2_log={mid_log_r2:.3f}")
    print(f"  Width: MAE={width_mae:,.0f}  RMSE={width_rmse:,.0f}  R2={width_r2:.3f}  R2_log={width_log_r2:.3f}")
    print(f"  Hit Rate: {overall['hit_rate']:.3f} (n={len(r['true_mid'])})")
    n_miss = int((~hit).sum())
    print(f"  DtB-MAE (misses only): RM {overall['dtb_mae']:,.0f} (n={n_miss})")
    print(f"  Width Ratio: median={ratio_median:.2f}  IQR=[{ratio_q1:.2f}, {ratio_q3:.2f}]")
    print(f"  Composite Score: {composite:.3f}")
    print(f"  Zero-width true rows (n={n_zero}): median pred_width={zero_pred_width_median:,.0f}")

    print("\n--- By Salary Bracket ---")
    for label, row in bracket_table.drop("Overall").iterrows():
        print(f"  [RM{label}] n={int(row['n'])}  MidMAE={row['mid_mae']:,.0f}  "
              f"HitRate={row['hit_rate']:.3f}  DtB={row['dtb_mae']:,.0f}  "
              f"WidthRatio(med)={row['width_ratio_median']:.2f}")

    print("\n--- By True-Width Bucket (Hit Rate + DtB-MAE) ---")
    edge_labels = {
        "narrow": f"narrow (RM0–{width_bucket_edges[0]:,.0f})",
        "medium": f"medium (RM{width_bucket_edges[0]:,.0f}–{width_bucket_edges[1]:,.0f})",
        "wide": f"wide (RM{width_bucket_edges[1]:,.0f}–∞)",
        "zero_width": "zero_width",
    }
    for bucket in ["narrow", "medium", "wide", "zero_width"]:
        row = width_bucket_table.loc[bucket]
        print(f"  [{edge_labels[bucket]}] n={int(row['n'])}  HitRate={row['hit_rate']:.3f}  DtB={row['dtb_mae']:,.0f}")
    print()

# Usage:
# width_edges = fit_width_bucket_edges(train_df)   # check this is ~[1000, 1500]
# print_report(test_preds, test_df, "BERT fine-tuned", "Test", width_edges)

In [ ]:
# ------------------------------------------------------------
# Custom collator: pad tokenizer fields via DataCollatorWithPadding,
# stack the [log_mid, log_width] labels separately (they are not
# per-token sequences and must not be passed through tokenizer.pad).
# ------------------------------------------------------------

def collate_fn(features):
    labels = [f.pop("labels") for f in features]
    batch = data_collator(features)  # pads input_ids/attention_mask only
    batch["labels"] = torch.stack([
        l if torch.is_tensor(l) else torch.tensor(l, dtype=torch.float)
        for l in labels
    ])
    return batch

In [ ]:
# ------------------------------------------------------------
# 7. TRAINING ARGUMENTS (plain MSE, val_loss for model selection)
# ------------------------------------------------------------
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=f"./{MODEL_NAME.split('/')[-1]}_seed{SEED}",
    learning_rate=0.00005,
    lr_scheduler_type="cosine",
    warmup_steps=1200,
    weight_decay=0.01,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    save_total_limit=2,
    dataloader_num_workers=2,
    remove_unused_columns=True,
    max_grad_norm=1.0,
    seed=SEED,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=make_compute_metrics(val_df),
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Stopped at epoch:", trainer.state.epoch)

Epoch,Training Loss,Validation Loss,Mid Mae,Mid Rmse,Mid R2,Mid Log R2,Width Mae,Width Rmse,Width R2,Width Log R2,Hit Rate,Dtb Mae,Width Ratio Median,Composite
1,0.415282,0.494011,1202.867048,1712.906822,0.494728,0.545891,482.732522,722.272680,0.390283,0.274223,0.330557,1015.848695,1.098510,0.260455
2,0.458017,0.423554,1020.256355,1540.846867,0.591138,0.663564,460.440465,710.885566,0.409356,0.366792,0.413139,885.113231,0.895796,0.299474
3,0.400003,0.451550,1128.247512,1584.727550,0.567519,0.601203,464.791069,698.377102,0.429959,0.333237,0.352996,945.596969,1.079813,0.279946
4,0.363475,0.470647,947.217994,1500.189050,0.612430,0.678334,516.313987,782.031969,0.285215,0.285598,0.449919,832.914264,0.776629,0.313244
5,0.274724,0.478962,1181.202696,1703.762323,0.500108,0.594950,462.240932,696.887291,0.432388,0.289046,0.344205,1010.738817,0.964803,0.252131


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Best checkpoint: ./bert-base-cased_seed42/checkpoint-2522
Stopped at epoch: 5.0


In [ ]:
m = trainer.evaluate()
print(m["eval_loss"], m["eval_hit_rate"])

Training Loss,Validation Loss,Epoch,Mid Mae,Mid Rmse,Mid R2,Mid Log R2,Width Mae,Width Rmse,Width R2,Width Log R2,Hit Rate,Dtb Mae,Width Ratio Median,Composite
0.274724,0.423656,5,1022.969826,1543.697711,0.589623,0.662709,459.666717,709.460644,0.411722,0.366799,0.411288,887.014412,0.899174,0.298647


0.4236564338207245 0.4112884570899838


In [ ]:
width_edges = fit_width_bucket_edges(train_df)
val_out = trainer.predict(val_dataset)  # set trainer.compute_metrics = None first
print_report(val_out.predictions, val_df, MODEL_NAME, "Validation", width_edges)

bert-base-cased — Validation (3-Layer Framework, RM2k brackets)

--- Overall ---
  Mid:   MAE=1,023  RMSE=1,544  R2=0.590  R2_log=0.663
  Width: MAE=460  RMSE=709  R2=0.412  R2_log=0.367
  Hit Rate: 0.411 (n=4323)
  DtB-MAE (misses only): RM 887 (n=2545)
  Width Ratio: median=0.90  IQR=[0.69, 1.25]
  Composite Score: 0.299
  Zero-width true rows (n=63): median pred_width=56

--- By Salary Bracket ---
  [RM0k-2k] n=265  MidMAE=685  HitRate=0.211  DtB=704  WidthRatio(med)=0.73
  [RM2k-4k] n=2207  MidMAE=738  HitRate=0.375  DtB=614  WidthRatio(med)=1.00
  [RM4k-6k] n=1198  MidMAE=914  HitRate=0.566  DtB=918  WidthRatio(med)=0.81
  [RM6k-8k] n=329  MidMAE=1,730  HitRate=0.331  DtB=1,259  WidthRatio(med)=0.81
  [RM8k-10k] n=188  MidMAE=2,010  HitRate=0.378  DtB=1,506  WidthRatio(med)=0.74
  [RM10k-12k] n=52  MidMAE=2,766  HitRate=0.385  DtB=2,618  WidthRatio(med)=0.73
  [RM12k-14k] n=47  MidMAE=3,714  HitRate=0.319  DtB=2,671  WidthRatio(med)=0.56
  [RM14k-16k] n=11  MidMAE=4,901  HitRate=0

/tmp/ipykernel_523/1937989366.py:140: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = df_eval.groupby("salary_bracket", observed=True).apply(_agg)
/tmp/ipykernel_523/1937989366.py:161: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_width_bucket)


In [ ]:
# ------------------------------------------------------------
# 9. TEST SET EVALUATION
# ------------------------------------------------------------

width_edges = fit_width_bucket_edges(train_df)
print("Width bucket edges (fit on train):", width_edges)

# compute_metrics is bound to val_df, so detach it for test prediction
trainer.compute_metrics = None
test_output = trainer.predict(test_dataset)
test_predictions = test_output.predictions
test_labels = test_output.label_ids

# Row-order sanity check: predictions must line up with test_df
assert len(test_predictions) == len(test_df)
assert np.allclose(np.expm1(test_labels[:, 0]), test_df["salary_mid"].values, rtol=1e-4), "Row order mismatch"

print_report(
    predictions=test_predictions,
    split_df=test_df,
    model_name=MODEL_NAME,
    split_name="Test Set",
    width_bucket_edges=width_edges,
)

Width bucket edges (fit on train): [1000. 1500.]


bert-base-cased — Test Set (3-Layer Framework, RM2k brackets)

--- Overall ---
  Mid:   MAE=1,127  RMSE=1,685  R2=0.578  R2_log=0.623
  Width: MAE=489  RMSE=723  R2=0.440  R2_log=0.266
  Hit Rate: 0.370 (n=4324)
  DtB-MAE (misses only): RM 988 (n=2726)
  Width Ratio: median=0.94  IQR=[0.70, 1.35]
  Composite Score: 0.275
  Zero-width true rows (n=114): median pred_width=623

--- By Salary Bracket ---
  [RM0k-2k] n=340  MidMAE=866  HitRate=0.100  DtB=831  WidthRatio(med)=1.59
  [RM2k-4k] n=2161  MidMAE=824  HitRate=0.333  DtB=711  WidthRatio(med)=1.04
  [RM4k-6k] n=1086  MidMAE=1,009  HitRate=0.551  DtB=1,081  WidthRatio(med)=0.83
  [RM6k-8k] n=353  MidMAE=1,697  HitRate=0.331  DtB=1,229  WidthRatio(med)=0.88
  [RM8k-10k] n=227  MidMAE=2,100  HitRate=0.374  DtB=1,672  WidthRatio(med)=0.75
  [RM10k-12k] n=56  MidMAE=2,414  HitRate=0.464  DtB=2,619  WidthRatio(med)=0.77
  [RM12k-14k] n=51  MidMAE=2,963  HitRate=0.353  DtB=1,712  WidthRatio(med)=0.56
  [RM14k-16k] n=20  MidMAE=5,267  HitRa

/tmp/ipykernel_523/1937989366.py:140: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = df_eval.groupby("salary_bracket", observed=True).apply(_agg)
/tmp/ipykernel_523/1937989366.py:161: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_width_bucket)
